# Detección de Phishing - CC421 Inteligencia Artificial (UNI-FC)
Avance Preliminar del Proyecto Final (75 %)

Notebook de experimentos y visualizaciones. Reutiliza los módulos de src/ para mantener una única fuente de verdad: el notebook documenta y visualiza, la lógica vive en el paquete.

Integrantes: Estacio Sanchez, Ortega Turpo, Lerzundi Ríos, Vega Bendezu, Iman Noriega.

## 0. Configuración y reproducibilidad

In [1]:
import sys, os, random
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
random.seed(config.RANDOM_STATE); np.random.seed(config.RANDOM_STATE)
print('Semilla fijada en', config.RANDOM_STATE)

Semilla fijada en 42


## 1. Carga y consolidación del corpus
Corpus público Enron-Spam (~33k correos). El loader deduplica y elimina correos vacíos.

In [2]:
from src.data_loader import load_corpus
df = load_corpus()
print('Correos:', len(df))
df[['subject','message','label','source']].head()

[data_loader] Corpus consolidado: 30462 correos (51 vacíos y 3203 duplicados removidos).
Correos: 30462


,subject,message,label,source
0,christmas tree farm pictures,,0,enron
1,"vastar resources , inc .","gary , production from the high island larger ...",0,enron
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,enron
3,re : issue,fyi - see note below - already done .\nstella\...,0,enron
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,enron


## 2. Limpieza y normalización del texto

In [3]:
from src.preprocess import clean_text, preprocess_series
ejemplo = 'URGENT! Verify your <b>account</b> at http://phish.example.com or email admin@bank.com. Call 1-800-555-0199!'
print('ANTES:', ejemplo)
print('DESPUÉS:', clean_text(ejemplo))
df['clean_text'] = preprocess_series(df['text'])
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print('Correos no vacíos tras limpieza:', len(df))

ANTES: URGENT! Verify your <b>account</b> at http://phish.example.com or email admin@bank.com. Call 1-800-555-0199!
DESPUÉS: urgent verify account url_token email email_token call num_token num_token num_token num_token


Correos no vacíos tras limpieza: 30461


## 3. Análisis exploratorio (EDA)
Distribución de clases, longitudes y palabras más frecuentes por clase.

In [4]:
from src import eda
print('Distribución de clases:', eda.class_distribution(df))
print('Estadísticas de longitud:', eda.length_stats(df))

Distribución de clases: {'total': 30461, 'legit': 15910, 'phishing': 14551, 'phishing_ratio': 0.4776927874987689}


Estadísticas de longitud: {'mean_tokens': 305.97212829519714, 'median_tokens': 154.0, 'p95_tokens': 935.0, 'max_tokens': 45450}


In [5]:
dist = eda.class_distribution(df)
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].bar(config.CLASS_NAMES, [dist['legit'], dist['phishing']], color=['#2a9d8f','#e76f51'])
ax[0].set_title('Distribución de clases'); ax[0].set_ylabel('Nº correos')
lengths = df['text'].str.split().map(len)
cap = int(lengths.quantile(0.99))
for lab,c,n in [(0,'#2a9d8f','legítimo'),(1,'#e76f51','phishing')]:
    ax[1].hist(lengths[df['label']==lab].clip(upper=cap), bins=40, alpha=0.6, color=c, label=n)
ax[1].set_title('Longitud por clase'); ax[1].set_xlabel('tokens'); ax[1].legend()
plt.tight_layout(); plt.show()

/tmp/claude-501/ipykernel_70469/2906989615.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Hallazgo de auditoría (importante). Las palabras más frecuentes de la clase legítima (enron, ect, hou, gas, energy) son artefactos de identidad corporativa: el ham proviene de una única fuente (correo interno de Enron). Esto es un confound de fuente única que infla las métricas y motiva integrar SpamAssassin y Nazario en la fase final.

In [6]:
tw = eda.top_words_by_class(df, 'clean_text', top_n=15)
fig, axes = plt.subplots(1,2,figsize=(11,5))
for ax,(name,color) in zip(axes, [('legit','#2a9d8f'),('phishing','#e76f51')]):
    items = tw[name][::-1]; w=[x for x,_ in items]; f=[c for _,c in items]
    ax.barh(w,f,color=color); ax.set_title(f'Top palabras - {name}')
plt.tight_layout(); plt.show()

/tmp/claude-501/ipykernel_70469/1412923951.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Partición estratificada y entrenamiento
Partición 70/15/15. El test se reserva y se evalúa una sola vez.

In [7]:
from src.features import stratified_split
from src.train import build_models, train_model
split = stratified_split(df['clean_text'].values, df['label'].values)
print({k: len(v) for k,v in split.items() if k.startswith('X')})
models = build_models(); trained = {}
for name, m in models.items():
    trained[name], _ = train_model(name, m, split['X_train'], split['y_train'])

{'X_train': 21322, 'X_val': 4569, 'X_test': 4570}


[train] naive_bayes: entrenado en 3.69s


[train] logistic_regression: entrenado en 4.49s


[train] random_forest: entrenado en 5.22s


[train] svm_linear: entrenado en 3.81s


## 5. Evaluación comparativa en validación

In [8]:
from src.evaluate import compute_metrics, _proba_positive
rows = []
scores = {}
for name, m in trained.items():
    met = compute_metrics(m, split['X_val'], split['y_val'], name)
    scores[name] = _proba_positive(m, split['X_val'])
    rows.append({k:met[k] for k in ['model','accuracy','precision','recall','f1','roc_auc','pr_auc']})
tabla = pd.DataFrame(rows).set_index('model').round(4)
tabla

,accuracy,precision,recall,f1,roc_auc,pr_auc
model,,,,,,
naive_bayes,0.9875,0.9872,0.9867,0.9869,0.9989,0.9987
logistic_regression,0.9880,0.9810,0.9940,0.9875,0.9991,0.9990
random_forest,0.9845,0.9783,0.9895,0.9838,0.9988,0.9986
svm_linear,0.9923,0.9913,0.9927,0.9920,0.9997,0.9996


In [9]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score, roc_auc_score
fig, ax = plt.subplots(1,2,figsize=(12,4.5))
for name, ys in scores.items():
    fpr,tpr,_ = roc_curve(split['y_val'], ys)
    ax[0].plot(fpr,tpr,label=f"{name} ({roc_auc_score(split['y_val'],ys):.3f})")
    pr,rc,_ = precision_recall_curve(split['y_val'], ys)
    ax[1].plot(rc,pr,label=f"{name} ({average_precision_score(split['y_val'],ys):.3f})")
ax[0].plot([0,1],[0,1],'k--',alpha=.4); ax[0].set_title('ROC'); ax[0].legend(fontsize=8)
ax[1].set_title('Precision-Recall'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

/tmp/claude-501/ipykernel_70469/836005947.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Modelo ganador y evaluación final en test
Seleccionamos por F1 en validación y reportamos test una sola vez.

In [10]:
from src.evaluate import meets_targets
best = tabla['f1'].idxmax()
print('Mejor modelo:', best)
test_met = compute_metrics(trained[best], split['X_test'], split['y_test'], best)
print('Metas cumplidas:', meets_targets(test_met))
{k:round(v,4) for k,v in test_met.items() if isinstance(v,float)}

Mejor modelo: svm_linear


Metas cumplidas: {'recall>=0.90': True, 'f1>=0.88': True, 'pr_auc>=0.92': True}


{'accuracy': 0.9904,
 'precision': 0.9877,
 'recall': 0.9922,
 'f1': 0.9899,
 'roc_auc': 0.9991,
 'pr_auc': 0.999}

In [11]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(trained[best], split['X_test'], split['y_test'],
    display_labels=config.CLASS_NAMES, cmap='Blues', values_format='d')
plt.title(f'Matriz de confusión (test) - {best}'); plt.show()

/tmp/claude-501/ipykernel_70469/2017269754.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(f'Matriz de confusión (test) — {best}'); plt.show()


## 7. Inferencia sobre correos en bruto

In [12]:
from src.predict import predict_email
correos = [
  'URGENT: Your PayPal account is limited. Click http://paypa1-verify.com/login to restore access now.',
  'Hi team, attaching the Q3 gas nomination schedule for review. Thanks, Daren',
]
for c in correos:
    r = predict_email(c, model=trained[best])
    print(f"[{r['label_name']:>9}] p={r['phishing_proba']:.3f} :: {c[:60]}...")

[ phishing] p=0.999 :: URGENT: Your PayPal account is limited. Click http://paypa1-...
[ legítimo] p=0.000 :: Hi team, attaching the Q3 gas nomination schedule for review...


## 8. Conclusiones preliminares
- Los cuatro clasificadores superan las metas (recall>=0.90, F1>=0.88, PR-AUC>=0.92) sobre Enron-Spam; SVM lineal es el mejor.
- Caveat honesto: el ham es de fuente única (Enron interno), lo que infla las métricas. La validez externa frente a spear-phishing real (Nazario) está pendiente.
- Siguiente 25 %: integrar SpamAssassin + Nazario, comparar TF-IDF vs embeddings, evaluar SMOTE, GridSearchCV e interpretabilidad.